In [ ]:
!pip install faster-whisper edge-tts moviepy==1.0.3
!sed -i 's/none/read,write/g' /etc/ImageMagick-6/policy.xml
!apt install imagemagick

In [ ]:
import os
import asyncio
import edge_tts
from faster_whisper import WhisperModel
from moviepy.editor import ColorClip, TextClip, CompositeVideoClip, AudioFileClip

# 1. SCRIPT (Wealth/Business/Psychology Fact)
TEXT = "Did you know that Apple doesn't make most of its money from selling iPhones? They make billions just by charging Google to be the default search engine on Safari. That is the real power of having an ecosystem."
VOICE = "en-US-ChristopherNeural" # Very charismatic American male voice
AUDIO_FILE = "voice.mp3"

# 2. GENERATE AUDIO (Edge-TTS)
async def generate_audio():
    communicate = edge_tts.Communicate(TEXT, VOICE)
    await communicate.save(AUDIO_FILE)

print("Generating audio...")
await generate_audio()

# 3. GET WORD TIMESTAMPS (Faster-Whisper)
print("Synchronizing subtitles with AI...")
model = WhisperModel("base", device="cuda", compute_type="float16") # Uses T4 GPU!
segments, info = model.transcribe(AUDIO_FILE, word_timestamps=True)

words_info = []
for segment in segments:
    for word in segment.words:
        words_info.append({"word": word.word, "start": word.start, "end": word.end})

# 4. COMPOSITE VIDEO (MoviePy)
print("Rendering video (Tiktok/Shorts Format)...")
audio_clip = AudioFileClip(AUDIO_FILE)
video_duration = audio_clip.duration

# Background (Black for now, can be replaced with Minecraft/GTA gameplay video)
bg_clip = ColorClip(size=(1080, 1920), color=(15, 15, 15)).set_duration(video_duration)

subtitle_clips = []
for word_data in words_info:
    # The word to display
    txt_clip = TextClip(word_data["word"].strip(), fontsize=100, color='yellow', font='Impact', stroke_color='black', stroke_width=3)
    
    # Timing the word display
    txt_clip = txt_clip.set_position('center').set_start(word_data["start"]).set_end(word_data["end"])
    subtitle_clips.append(txt_clip)

# Overlay all clips
final_video = CompositeVideoClip([bg_clip] + subtitle_clips)
final_video = final_video.set_audio(audio_clip)

# Render!
final_video.write_videofile("final_shorts.mp4", fps=24, codec="libx264", audio_codec="aac")
print("✅ VIDEO READY! Check final_shorts.mp4 in the files tab on the left.")
